# 动态规划、蒙特卡洛与时序差分 (DP vs MC vs TD)


理解这三种求解价值函数的核心算法，关键在于搞懂两个维度：
1. **需不需要环境的绝对上帝视角？**（是否已知环境模型/是否需要采样）
2. **更新时是“走到底再算总账”，还是“走一步看一步”？**（是否自举 Bootstrapping）

---

## 1. 动态规划 (Dynamic Programming, DP)

* **核心特点**：**已知模型、全量更新、走一步看一步。**
* **前提条件**：必须完全掌握环境的运转规则（即状态转移概率 $P$ 和奖励函数 $R$），属于“白盒”环境。
* **计算方式**：不需要与环境实际交互，而是在脑海中遍历所有可能的分支计算期望。
* **核心公式**（DP 策略评估）：
  $$V_{k+1}(s) \leftarrow \sum_a \pi(a \mid s) \left[ R(s, a) + \gamma \sum_{s'} P(s' \mid s, a) V_k(s') \right]$$
  * **解析**：利用已知的世界模型 $P(s' \mid s, a)$ 进行完美推演，并用下一步的估计值 $V_k(s')$ 更新当前的 $V_{k+1}(s)$，这称为**自举（Bootstrapping）**。

## 2. 蒙特卡洛 (Monte Carlo, MC)

* **核心特点**：**未知模型、真实采样、走到底算总账。**
* **前提条件**：无需知道环境模型（黑盒环境），通过在环境中不断试错、采样完整轨迹来积累经验。
* **计算方式**：必须把整个回合（Episode）走完，拿到最终的真实总回报（Return, $G_t$），再回溯更新沿途的状态价值。
* **核心公式**（MC 更新）：
  $$V(s) \leftarrow V(s) + \alpha [G_t - V(s)]$$
  * **解析**：$G_t$ 是整条轨迹跑完后的真实完整回报。由于**没有自举**，结果无偏（Unbiased），但因为每局轨迹差异大，方差（Variance）较高。

## 3. 时序差分 (Temporal Difference, TD)

* **核心特点**：**未知模型、真实采样、走一步看一步。**
* **算法地位**：结合了 DP 和 MC 的优点，是现代强化学习（如 Q-Learning）的基石。
* **计算方式**：不需要环境模型（实机采样），且**不需要等回合结束**。走一步拿到即时奖励 $r$，结合对下一步的“旧估计” $V(s')$，立刻更新当前状态价值。
* **核心公式**：
  * **TD(0) 更新**：
    $$V(s) \leftarrow V(s) + \alpha [r + \gamma V(s') - V(s)]$$
  * **TD Error**（TD 误差）：
    $$\delta = r + \gamma V(s') - V(s)$$
  * **解析**：用 $r + \gamma V(s')$（即 TD 目标）代替 MC 中的真实回报 $G_t$。深度强化学习的核心通常就是最小化这个 TD 误差 $\delta$。

---

## 💡 核心总结对比

| 算法 | 是否需要环境模型？(Model-free?) | 更新深度 (是否自举 Bootstrapping?) | 优缺点对比 |
| :--- | :--- | :--- | :--- |
| **DP** | ❌ **需要** (Model-based) | ✅ **走一步看一步** (自举) | 理论完美，但现实中极难获知精准模型，且存在维度灾难。 |
| **MC** | ✅ **不需要** (无模型采样) | ❌ **走到结局才更新** (无自举) | 无偏，但方差大（单次采样随机性高），且必须等回合结束才能更新。 |
| **TD** | ✅ **不需要** (无模型采样) | ✅ **走一步看一步** (自举) | 支持在线实时更新，方差较小。结合了 DP 和 MC 的优点，是 RL 主流。 |

值得注意：这里虽然R(s,a) 维护的是一个奖励表格，不同状态S不同动作a，带来的奖励都不同，但是DP是都知道的是有这一张表格的，但是TD是不需要维护这么一张表格的
所以DP是需要环境模型，但是TD不需要的
---

## 🚗 形象的比喻：预测通勤时间

* **动态规划 (DP)**：你有完美的城市交通流控图表，知道每个路口绿灯和拥堵的精确概率，在家拿笔算出各条路线的数学期望。
* **蒙特卡洛 (MC)**：你不看图表，亲自开车跑，到了公司看手表记下总时间（算总账）。跑100次取平均值。
* **时序差分 (TD)**：你开车上路，刚过一个拥堵路口花了 10 分钟。你看了看剩下的路程，心想“按过去经验，剩下的路大概还要 20 分钟”，于是你立刻在车上调整了今天的总时间预测（10+20=30分钟）。

# 三者关键区别（非常重要）

都要维系一个 v 价值表，最开始因为不知道整个价值表，所以说他会使用一个估计价值来替代未知的真实的价值，而三者的区别就在于如何算这个估计价值 target

DP下一步使用估计的，MC是一口气全部用真实的算出来， TD是下一步用真实的 + 下下步后面的全部使用估计

三种经典方法的核心差异，就在于如何构造这个 **target**。

### 1. DP（动态规划）
DP 假设已知完整的环境模型。它利用贝尔曼期望方程，把策略下所有动作分支和下一状态分支全部展开，直接计算期望。因为模型已经给出了所有分支的概率和奖励，不需要实际进入环境采样，遍历一遍状态表即可完成一轮更新。

### 2. MC（蒙特卡洛）
MC 不假设知道模型。它等到一次完整的 episode 结束后，把实际发生的总回报 $G_t$ 直接当作 target。$G_t$ 是这条轨迹上从该状态出发的真实回报样本，不是模型算出的平均值。它的更新只能在 episode 结束后进行。

### 3. TD（时序差分）
TD 同样不知道模型，但它不等 episode 结束。每走一步，它就把当前观察到的即时奖励 $R_{t+1}$ 和下一状态的当前估计值 $V(S_{t+1})$ 组合成 target，即 $R_{t+1} + \gamma V(S_{t+1})$。

TD 能这样做的原因是回报具有递归结构：

$$
G_t = R_{t+1} + \gamma G_{t+1}
$$

$G_{t+1}$ 是从下一状态开始的完整回报。在 $t+1$ 时刻，$G_{t+1}$ 尚未发生，TD 就用表里对下一状态的当前估计 $V(S_{t+1})$ 代替它：

$$
\text{target}_{\text{TD}} = R_{t+1} + \gamma V(S_{t+1})
$$

这一步叫**自举（bootstrapping）**。它让 TD 能一步一更新，也会带来后面要讨论的偏差。

---

### 统一更新框架

三种方法虽然 target 不同，但更新都可以归入同一个框架：把当前估计向 target 移动。

$$
V(s) \leftarrow V(s) + \alpha \big[ \text{target} - V(s) \big]
$$

- **$V(s)$**：表里的旧数字。
- **target**：这次算出来的新估计。
- **$\alpha$**：控制步幅。

$\text{target} - V(s)$ 表示“这次认为旧表错了多少”；乘上 $\alpha$，就是这次实际改多少。$\alpha=1$ 时直接覆盖旧值，$\alpha$ 较小时只往目标方向挪一小步。

> **总结**：本节的重点不是分别学三种算法，而是比较三种构造 target 的方式。三种方法都在改同一张价值表，区别只在 target 的来源。

直接阅读这个帖子 ： https://walkinglabs.github.io/hands-on-modern-rl/chapter03_mdp/dp-mc-td

TD误差 = [ 一步贝尔曼目标 ($r + \gamma V(s')$) ] - [ 价值估计 ($V(s)$) ]

而DP是知道环境模型，知道在每个环境下做什么动作可以切换到下一个环境，这个情况下就可以枚举，使用DP

不知道环境模型使用MC 和 TD， MC是全程随机动作，然后拿到奖励，再用奖励总和来更新当前状态的 V(s)。多次尝试以后拿到多个VS来跟来不断逼近真实的VS。

Td 是用当前的这一步真实奖励和下一步的一个预估 V_s 来更新当前的 V_s，这样就不用从头到尾更新

In [ ]:
import random

STATES = ["S", "M", "G"]
GAMMA = 1.0


def step(state, action):
    # 环境本身是确定的：给定状态和动作，下一状态、奖励都固定。
    # 随机性只来自后面的策略 sample_action()。
    if state == "S":
        return ("M", -1) if action == "right" else ("S", -2)
    if state == "M":
        return ("G", -1) if action == "right" else ("S", -2)
    return "G", 0


def sample_action():
    # 固定策略 pi：80% 向右，20% 向左。
    return "right" if random.random() < 0.8 else "left"


def dp_policy_evaluation(n_iter=1_000):
    V = {s: 0.0 for s in STATES}
    for _ in range(n_iter):
        # DP 知道模型，因此可以直接枚举两个动作分支。
        # 这里用 old 做同步更新：本轮读旧表，写出新表。
        old = V.copy()
        V["S"] = 0.8 * (-1 + GAMMA * old["M"]) + 0.2 * (-2 + GAMMA * old["S"])
        V["M"] = 0.8 * (-1 + GAMMA * old["G"]) + 0.2 * (-2 + GAMMA * old["S"])
        V["G"] = 0.0
    return V


def generate_episode():
    # MC 和 TD 都不知道模型，只能让智能体真的走一局。
    episode = []
    state = "S"
    while state != "G":
        action = sample_action()
        next_state, reward = step(state, action)
        episode.append((state, reward, next_state))
        state = next_state
    return episode


def mc_every_visit(n_episodes=1_000_000, seed=0):
    random.seed(seed)
    V = {s: 0.0 for s in STATES}
    N = {s: 0 for s in STATES}
    for _ in range(n_episodes):
        episode = generate_episode()
        G = 0.0
        # MC 等整局结束后，从后往前累加完整回报 G_t。
        for state, reward, _ in reversed(episode):
            G = reward + GAMMA * G
            N[state] += 1
            # 每次访问都更新；1/N 是样本平均的增量写法。
            V[state] += (G - V[state]) / N[state]
    return V


def td_zero(n_episodes=1_000_000, seed=0):
    random.seed(seed)
    V = {s: 0.0 for s in STATES}
    N = {s: 0 for s in STATES}
    for _ in range(n_episodes):
        state = "S"
        while state != "G":
            action = sample_action()
            next_state, reward = step(state, action)
            N[state] += 1
            alpha = 1.0 / N[state]
            # TD 不等整局结束：一步奖励 + 下一状态当前估计。
            target = reward + GAMMA * V[next_state]
            V[state] += alpha * (target - V[state])
            state = next_state
    return V


def show(name, values):
    print(f"{name}: S={values['S']:.6f}, M={values['M']:.6f}, G={values['G']:.6f}")


def summarize(name, runs):
    mean_s = sum(v["S"] for v in runs) / len(runs)
    mean_m = sum(v["M"] for v in runs) / len(runs)
    min_s, max_s = min(v["S"] for v in runs), max(v["S"] for v in runs)
    min_m, max_m = min(v["M"] for v in runs), max(v["M"] for v in runs)
    print(
        f"{name}: mean S={mean_s:.6f} [{min_s:.6f}, {max_s:.6f}], "
        f"mean M={mean_m:.6f} [{min_m:.6f}, {max_m:.6f}]"
    )


print("single run")
show("DP", dp_policy_evaluation())
show("MC", mc_every_visit(seed=0))
show("TD", td_zero(seed=0))

print("\n5-run summary")
seeds = range(5)
summarize("MC", [mc_every_visit(seed=s) for s in seeds])
summarize("TD", [td_zero(seed=s) for s in seeds])